# 06 — Finding features from errors

The feature loop, starting at exp_0005 as promised: **segment the OOF errors →
hypothesise → one experiment → verdict.** Features come from where the model fails,
not from imagination — plus the one theory-driven source, arithmetic a tree cannot
build (LEARNING.md: ratios, weak-pair interactions, cross-row aggregations).

Both experiments in this notebook came back negative. That is not a failed notebook —
a negative with a mechanism is a crossed-off idea that never gets retried in week 3,
and the error profile that motivated it still constrains what *could* work.

In [1]:
%load_ext autoreload
%autoreload 2

import warnings

warnings.filterwarnings("ignore")

import numpy as np
import polars as pl

from s6e7 import cv, eda, io

pl.Config.set_tbl_rows(15)
pl.Config.set_tbl_width_chars(200)
train = io.load_train()
oof_base = np.load(cv.OOF_DIR / "exp_0001.npy")

---
## 1. Where does the baseline fail?

`eda.recall_by_bin` slices the OOF errors along one feature at a time. The first row
is the global reference; a bin whose recall craters below it marks a segment the
current features cannot express. `sleep_duration` — the strongest single feature —
is the standout:

In [2]:
eda.recall_by_bin(train, oof_base, "sleep_duration", io.TARGET, n_bins=8).select(
    "bin", "n_rows", "recall_at-risk", "recall_fit", "recall_unhealthy", "balanced_acc"
)

bin,n_rows,recall_at-risk,recall_fit,recall_unhealthy,balanced_acc
str,i64,f64,f64,f64,f64
"""all""",690088,0.9927,0.8254,0.8007,0.8729
"""3 - 5.63""",76054,0.9876,0.0488,0.9083,0.6482
"""5.63 - 6.16""",77375,0.9858,0.0188,0.8681,0.6242
"""6.16 - 6.62""",74340,1.0,0.0,0.0,0.3333
"""6.62 - 6.99""",79158,0.9999,0.009,0.0,0.3363
"""6.99 - 7.37""",75669,0.9964,0.8613,0.0,0.6193
"""7.37 - 7.81""",76910,0.997,0.8711,0.0,0.6227
"""7.81 - 8.39""",77616,0.997,0.871,0.0058,0.6246
"""8.39 - 10""",76967,0.9962,0.8747,0.005,0.6253


Three readings:

- **The minorities live at the extremes.** `fit` recall is ~0 below 6.2 hours and
  ~0.87 above 7; `unhealthy` is the mirror image. In the middle band (6.2–7.0) *both*
  minority recalls are ~0 — where sleep is uninformative, the other 12 features barely
  separate the classes at all.
- **The null bin is a crater.** When `sleep_duration` is missing, `unhealthy` recall
  falls 0.80 → 0.21. The model leans on sleep so hard that its absence takes the
  minorities down with it.
- `at-risk` never suffers — the majority class is the default answer everywhere.

The activity pair shows the same pattern for `fit`:

In [3]:
for col in ("step_count", "exercise_duration"):
    t = eda.recall_by_bin(train, oof_base, col, io.TARGET, n_bins=8)
    print(t.select("bin", "n_rows", "recall_fit", "balanced_acc").head(4))

shape: (4, 4)
┌─────────────┬────────┬────────────┬──────────────┐
│ bin         ┆ n_rows ┆ recall_fit ┆ balanced_acc │
│ ---         ┆ ---    ┆ ---        ┆ ---          │
│ str         ┆ i64    ┆ f64        ┆ f64          │
╞═════════════╪════════╪════════════╪══════════════╡
│ all         ┆ 690088 ┆ 0.8254     ┆ 0.8729       │
│ 1002 - 3378 ┆ 84407  ┆ 0.02       ┆ 0.6049       │
│ 3378 - 5389 ┆ 84396  ┆ 0.1661     ┆ 0.6518       │
│ 5389 - 7150 ┆ 84760  ┆ 0.7511     ┆ 0.8504       │
└─────────────┴────────┴────────────┴──────────────┘


shape: (4, 4)
┌─────────────┬────────┬────────────┬──────────────┐
│ bin         ┆ n_rows ┆ recall_fit ┆ balanced_acc │
│ ---         ┆ ---    ┆ ---        ┆ ---          │
│ str         ┆ i64    ┆ f64        ┆ f64          │
╞═════════════╪════════╪════════════╪══════════════╡
│ all         ┆ 690088 ┆ 0.8254     ┆ 0.8729       │
│ 0 - 22.2    ┆ 84500  ┆ 0.0052     ┆ 0.5988       │
│ 22.2 - 29.2 ┆ 86244  ┆ 0.1608     ┆ 0.6517       │
│ 29.2 - 34.5 ┆ 84912  ┆ 0.747      ┆ 0.8476       │
└─────────────┴────────┴────────────┴──────────────┘


---
## 2. Two hypotheses, one experiment each

**H1 — explicit missingness indicators (exp_0005).** The null-bin craters plus the
known `bmi_is_null` signal (unhealthy 2.79% vs 8.47%, ~22σ) suggest handing the model
13 explicit is-null flags. Counter-argument, known in advance: LightGBM already routes
NaN at every split, so the flags may be redundant.

**H2 — activity-intensity ratios (exp_0006).** `fit` recall dies at low steps and low
exercise minutes, and the activity trio is internally correlated (0.37–0.44). A
*ratio* (calories per step, per exercise-minute) is exactly the diagonal combination
a tree cannot build from axis-aligned splits — if intense-but-brief exercisers are
hiding in the low bins, ratios expose them.

In [4]:
from s6e7.cv import ExperimentConfig

for exp_id, feats, changed in [
    ("exp_0005", "indicators", "add 13 missingness indicator columns"),
    ("exp_0006", "ratios", "add 3 activity-intensity ratios (from exp_0001 error profile)"),
]:
    r = cv.run(ExperimentConfig(exp_id, "lgbm", features=feats, parent="exp_0001", changed=changed),
               train=train, test=io.load_test(), if_logged="skip")
    print(f"{exp_id}: cv {r.cv_mean:.5f} ± {r.cv_std:.5f}")

exp_0005: cv 0.87291 ± 0.00205


exp_0006: cv 0.87285 ± 0.00200


## 3. Verdicts — paired, as always

In [5]:
cv.paired_diff("exp_0005", "exp_0001", train=train)

fold,exp_0005,exp_0001,diff,t
str,f64,f64,f64,null
"""0""",0.87121,0.87121,0.0,null
"""1""",0.87515,0.87515,0.0,null
"""2""",0.87502,0.87502,0.0,null
"""3""",0.87089,0.87089,0.0,null
"""4""",0.87229,0.87229,0.0,null
"""mean""",0.87291,0.87291,0.0,null


In [6]:
flips = int((np.load(cv.OOF_DIR / "exp_0005.npy").argmax(1) != oof_base.argmax(1)).sum())
moved = float(np.abs(np.load(cv.OOF_DIR / "exp_0005.npy") - oof_base).max())
print(f"probabilities moved by up to {moved:.3f}, argmax decisions flipped: {flips:,} of {train.height:,}")

probabilities moved by up to 0.016, argmax decisions flipped: 0 of 690,088


**H1 is a perfect zero — with a mechanism.** The indicator columns nudged the
probabilities (the trees *did* use them occasionally) but flipped essentially no
decisions: NaN routing already encodes everything the flags say. This is the cheapest
kind of negative — theory said "probably redundant", the experiment made it a fact.

**H2 missed.** The paired t is −0.3: the low-activity `fit` rows are apparently not
intense-but-brief exercisers; the ratio adds no separating direction:

In [7]:
cv.paired_diff("exp_0006", "exp_0001", train=train)

fold,exp_0006,exp_0001,diff,t
str,f64,f64,f64,f64
"""0""",0.8709,0.87121,-0.00031,null
"""1""",0.87457,0.87515,-0.00058,null
"""2""",0.87505,0.87502,0.00003,null
"""3""",0.87076,0.87089,-0.00013,null
"""4""",0.87298,0.87229,0.00069,null
"""mean""",0.87285,0.87291,-0.00006,-0.3


---
## 4. What the failures teach

The error profile is unchanged: every remaining error lives where the raw features
carry no class signal (mid-band sleep, missing sleep). A useful feature would have to
separate `fit` from `unhealthy` *inside* those segments, and none of the 13 columns —
nor their arithmetic — does. On synthetic playground data with no entities to
aggregate over, that is usually the end of the feature road; the honest next moves are
prior correction and ensembling (notebook 07), not a fourth feature idea.

The loop to carry to the next competition: **profile errors → name the segment → one
hypothesis per experiment → paired verdict → log it either way.** Random feature
ideas skip the first two steps, and that is why they cost weeks.